# LLM tool calling

LLMs are good at interpreting and conversate in natural language, but there are things they cannot handle (up to date information such as news, current weather, etc.) simply because it was not (could not be) part of their training data, or things like mathematical computations which is _not_ a good fit for natural language.

Thus, LLM can ask for help from _tools_ in those cases.

## How is tool calling performed in practice?

The process is best illustrateted with a minimal example.
As usual, we need to connect to the LLM service and make sure to pick a model that _supports tool calling._
To check if a model supports tool calling you can e.g. use the ollama site:
<https://ollama.com/search?c=tools>

In [ ]:
!pip -q install ollama

In [ ]:
from ollama import Client

OLLAMA_HOST = 'http://10.129.20.4:9090'
OLLAMA_MODEL = 'llama3.1:latest'

client = Client(host=OLLAMA_HOST)

In addition we need to provide a _tool_. It can be as simple as the following function:

In [ ]:
def add_numbers(a: int, b: int) -> int:
    """
    Add numbers

    Args:
      a (int): The first number
      b (int): The second number

    Returns:
      int: The sum of the two numbers
    """
    return a + b

Here, the name of the tool is not really important, but the type annotations and the docstring is. Ollama use python's introspection capabilities to analyze the function, but type annotation and documentation helps in getting the desired result.

We communicate with the model, via Ollama, by keeping a _message history_, a simple list of entries keeping track of what is going on. The entry _roles_ alternates between 'assistant' (the model) and 'user'/'tool' (you). The model responds to the last message in the list, keeping any prior messages as _context_ for the conversation. 

Initially, the message history contains just our query:

In [ ]:
messages = [{'role': 'user', 'content': 'What is three plus one?'}]

Let's add a little helper function that lets us examine the message history in a convenient format.

In [ ]:
import json
from IPython import display

def show_messages(messages):
    """Helper function to view messages in a nice format"""
    s = json.loads(json.dumps(messages, default=dict))
    return display.JSON(s, root="Message history")

Calling it will present the message history as a list whose entries that can be shown/hidden by clicking the triangles. Try it. For now, focus on the role and content fields, and ignore the rest.

In [ ]:
show_messages(messages)

We can now kick off the conversation with the LLM by passing the first message and a list of our tools to the client:

In [ ]:
messages = [{'role': 'user', 'content': 'What is three plus one?'}]

response = client.chat(
    OLLAMA_MODEL,
    messages=messages,
    tools=[add_numbers],
)
messages.append(response.message)

In [ ]:
show_messages(messages)

The only difference to a normal query is the additional parameter `tools` which is given a list of helpers appropriate for the query.

Instead of responding with an answer to the query, the LLM _may_<sup>1</sup> respond with details of what tool to call and the arguments for the call, like follows:

```python
print(response.message.tool_calls)
```

```
[ToolCall(function=Function(name='add_numbers', arguments={'a': 3, 'b': 1}))]
```

Nested in that message is the name of the tool to call and the required arguments.

---
<p><small>(1) There is no guarantee that a tool will be called in the code above, we cannot _force_ tool use.

If the last entry in the message history above does not contains a tool call, please re-run the cell until it does.</p></small>

---


We then have to call the tool with the given arguments on behalf of Ollama and feed the result back to the LLM, as part of the messages list, for the final response to be generated.

In [ ]:
result_of_toolcall = add_numbers(3, 1)

Now, the type of the result is `int` but the message history expects a string, so we'll use `str()` below to convert the reult into a string.

In [ ]:
messages.append({'role': 'tool', 'content': str(result_of_toolcall), 'name': "add_numbers"})

response = client.chat(
  OLLAMA_MODEL,
  messages=messages
)
messages.append(response.message)

In [ ]:
show_messages(messages)

With that we can print the answer to the query:

In [ ]:
print(messages[-1].content)

The process is shown in the figure below
```mermaid
flowchart LR
    user_prompt("User prompt"):::start_stop
    llm["LLM"]
    parsing["Harness"]
    decision{"Tool use?"}
    response["Answer"]
    tool_call["Tool call"]
    run["Run tool"]
    state["Add result to<br/>message history"]


    user_prompt --> llm

    subgraph Ollama internal
        llm --> parsing --> decision
    end

    decision -- NO --> response
    decision -- YES --> tool_call --> run --> state --> llm

    classDef start_stop fill: #9f6, stroke: #333, stroke-width:2px;
    classDef model fill: #f96, stroke: #333, stroke-width:1px;
```

Note that ollama will pre-process the LLM's response (in "Harness") and potentially issue a tool call. Tool calling capability is not a function of the model per se<sup>2</sup>, but a feature of ollama (or other model runners). We'll looe more into the details in the 'LLM-Agent' tutorial.

---
<p><small>(2) Tool using models are fine-tuned to convey their intent to call tools to the runner by special tokens, but as we shall see in the 'LLM-Agent' tutorial, that is not a strict requirement.</small></p>

---

#### A sidenote on messages and roles

If you find the usage of `messages` and `role` confusing, dont despair, have a look at this [notebook](./LLM-role-context.ipynb) for a longer explanation.

### Automating tool use

Of course, we want to streamline the use of tools, and one thing missing to fully automate the process is a mapping from tool names to the corresponding functions. The simplest solution is a dictionary for look-up:

In [ ]:
available_tools = {"add_numbers": add_numbers}

From that mapping we can get a list of tools:

In [ ]:
tools = list(available_tools.values())

### Error handling

So far, we have only explored the "happy path" but plenty of things can go wrong.  
The LLM might not issue a tool call at all, but even if it does

1. The LLM can hallucinate and request a non-existing tool
2. The LLM can pass the wrong parameters, or parameters of the wrong type
3. The tool can fail during execution

Let's approach all the above error cases by simply looping any error messages back to the model, in the hope that it can take the error into account and do better next time around. 

Besides the possible errors, we need to handle the fact the message history requires a string whereas the tool result type varies with the tool.

The simplest solution is function that we will call `tool_client` that does all this, and it makes use of a helper function `post_process_tool_call` that converts the result to a string (naively hardcoded for now and not suitable for general tools).

In [ ]:
from ollama import Client

def post_process_tool_call(result):
    return str(result)

def tool_client(query, available_tools):
    
    OLLAMA_HOST = 'http://10.129.20.4:9090'
    OLLAMA_MODEL = 'llama3.1:latest'
    client = Client(host=OLLAMA_HOST)
    tools = list(available_tools.values())

    messages = [{'role': 'user', 'content': query}]
    
    response = client.chat(OLLAMA_MODEL, messages=messages, tools=tools)
    messages.append(response.message)

    if not response.message.tool_calls:
        print("\n=== No tool call issued ===\n")

    while True:
        # If there was no tool call, we're done => stop processing
        if not response.message.tool_calls:
            break

        try:
            # Assume just a single tool call is issued
            tool = response.message.tool_calls[0]
            tool_name = tool.function.name or ""
            if tool_name not in available_tools:
                raise Exception("Error: Unknown tool")
            fcn = available_tools[tool_name]
            args = tool.function.arguments
            optional_args = args.pop("kwargs", {})
            if not isinstance(optional_args, dict):
                optional_args = {}
            # Call the tool
            result_of_toolcall = fcn(**args, **optional_args)
            print(f"Called tool:\n  {tool_name}({args}, {optional_args}) -> {result_of_toolcall}")
            tool_result = post_process_tool_call(result_of_toolcall)
        except Exception as e:
            tool_result = str(e)


        # Feed the tool result to the LLM in the message history
        messages.append({'role': 'tool', 'content': tool_result, 'name': tool_name})
        print("Sending result of tool call back to LLM...")

        # Get an updated response from the model
        response = client.chat(OLLAMA_MODEL, messages=messages, tools=tools)
        messages.append(response.message)

    return messages

The function takes a query and a dictionary of available tools as input

In [ ]:
messages = tool_client('What is three plus one?', available_tools)
print("---")
print(messages[-1].content)
show_messages(messages)

If the output says "=== No tool call issued ===", re-run the cell until it doesn't.

## Calling REST services

We are not limited to home grown functions to use as tools, and we can easily make use of e.g. online REST services by using python modules like `requests`.

In [ ]:
import requests
# help (requests.request)

We'll use the freely available [meowfacts](https://github.com/wh-iterabb-it/meowfacts) service residing on <https://meowfacts.herokuapp.com/> to fetch random cat facts for the following example.

In [ ]:
!curl https://meowfacts.herokuapp.com/

The request method provides enough information in its `__doc__` field (try `print(requests.request.__doc__)`) for the LLM to figure out the _schema_, i.e. the semantics of required (and optional) arguments and their types.

Unfortunately, `requests.request` returns a `requests.models.Response`

In [ ]:
result = requests.request(method='GET', url='https://meowfacts.herokuapp.com')
type(result)

Thus, we need to modify `post_process_tool_call` to use the `text` property of the result to get a string:

In [ ]:
def post_process_tool_call(result):
    try: 
        return result.text
    except:
        return str(result)

In [ ]:
messages = tool_client('Return a cat fact from https://meowfacts.herokuapp.com', {"request": requests.request})
print(messages[-1].content)
show_messages(messages)

As always, the answer varies from query to query, but you should get a response similar to:

> Here's a fun fact about cats: Your cat recognizes your voice, but it might just pretend not to care because of its cool nature.

## Summary

Tool calling is useful to get up-to-date information, or helping out with tasks that are hard for LLMs to get right. A tricky part is making sure the result returned from the tool is properly converted to a string as required by the LLM API. In this tutorial we have hardcoded that part for our tools, but a more generic solution would be to keep that functionality with each tool in a (slightly more complex) `available_tools`-mapping. However, as we will see in later tutorials, there are other appraoches.

Tool calling is typically not used stand-alone, but as the basis for _agents_ which we will turn to in the next tutorial ('LLM-Agents').